# A.22 Imported functions

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Sometimes it is convenient to express models with the help of functions that are not built into
AMPL. AMPL has facilities for importing functions and optionally checking the consistency of
their argument lists. Note: The practical details of using imported functions are highly systemdependent.
This section is concerned only with syntax; specific information will be found in
system-specific documentation, e.g., on the AMPL web site.

An imported function may need to be evaluated to translate the problem; for instance, if it plays
a role in determining the contents of a set, AMPL must be able to evaluate the function. In this case
the function must be linked, perhaps dynamically, with AMPL. On the other hand, if an imported
function's only role is in computing the value of a constraint or objective, AMPL never needs to
evaluate the function and can simply pass references to it on to a (nonlinear) solver.

Imported functions must be declared in a function declaration before they are referenced.
This statement has the form

```ampl
function name alias opt (domain - spec) opt type opt [ pipe litseq opt [ format fmt ] ] ;
```

in which name is the name of the function, and domain-spec amounts to a function prototype:

```ampl
domain-spec:
     domain-list
     ...
     nonempty-domain-list , ...
```

A domain-list is a (possibly empty, comma-separated) list of set expressions, asterisks (∗'s), direction
words (IN, OUT, or INOUT), direction words followed by set expressions, and iterated-domain-lists:

```ampl
iterated-domain-list:
       indexing ( nonempty domain-list )
```

An iterated-domain-list is equivalent to one repetition of its domain-list for each member in the
indexing set, and the domain of dummy variables appearing in the indexing extends over that
domain-list. The direction words indicate which way information flows: into the function (IN), out
of the function (OUT), or both, with IN the default. In a function invocation, OUT arguments are
assigned values specified by the function at the end of the command invoking the function.

Omitting the optional (domain-spec) in the function declaration is the same as specifying
(...). The function must be invoked with at least or exactly as many arguments as there are sets
in the domain-spec (after iterated-domain-lists have been expanded), depending on whether or not
the domain-spec ends with .... AMPL checks that each argument corresponding to a set in the
domain-list lies in that set. A * by itself in a domain-list signifies no domain checking for the corresponding
argument.

A function whose return value is not of interest can be invoked with a call command:

```ampl
call funcname( arglist );
```

Type can be symbolic or random or both; symbolic means the function returns a literal
(string) value rather than a numeric value, and random indicates that the "function" may return
different values for the same arguments, i.e., AMPL should assume that each invocation of the function
returns a different value.

The commands

```ampl
load       libnames opt ;
unload     libnames opt ;
reload     libnames opt ;
```

load, unload, or reload shared libraries (from which functions and `table` handlers are imported); libnames
is a comma-separated list of library names. When at least one libname is mentioned in the
load and unload commands, $AMPLFUNC is modified to reflect the full pathnames of the currently
loaded libraries. The reload command first unloads its arguments, then loads them. This
can change the order of loaded libraries and affect the visibility of imported functions: the first
name wins. With no arguments, load loads all the libraries currently in $AMPLFUNC; unload
unloads all currently loaded libraries, and reload reloads them (which is useful if some have
been recompiled).

The keyword pipe indicates that this is a pipe function, which means AMPL should start a separate
process for evaluating the function. Each time a function value is needed, AMPL writes a line
of arguments to the function process, then reads a line containing the function value from the process.
(Of course, this is only possible on systems that allow multiple processes.) A litseq is a
sequence of one or more adjacent literals or parenthesized string expressions, which AMPL concatenates
and passes to the operating system (i.e., to $SHELL) as the description of the process to
be invoked. In the absence of a litseq, AMPL passes a single literal, whose value is the name of the
function. If the optional format fmt is present, fmt must be a format, suitable for printf, that
tells AMPL how to format each line it sends to the function process. If no fmt is specified, AMPL
uses spaces to separate the arguments it passes to the pipe function.

For example:

```ampl
ampl: function mean2 pipe "awk '{print ($1+$2)/2}'";
ampl: display mean2(1,2) + 1;
mean2(1, 2) + 1 = 2.5
```

The function mean2 is expected (by default) to return numeric values; AMPL will complain if it
returns a string that does not represent a number.

The following functions are symbolic, to illustrate formatting and the passing of arguments.

```ampl
ampl: function f1 symbolic pipe "awk '" '{printf "x%s\n", $1}' "'";
ampl: function g1 symbolic pipe 'awk "{printf "XX%s\n", $1}"';
ampl: function cat symbolic pipe format ">>%s<<\n";
ampl: display f1(2/3);
f1(2/3) = x0.66666666666666667
ampl: display g1('abc');
g1('abc') = XXabc
ampl: display cat('some words');
cat('some words') = ">>'some words'<<"
```

The declaration of f1 specifies a litseq of 3 literals, while g1 specifies one literal; cat, having an
empty litseq, is treated as though its litseq were 'cat'. The literals in each litseq are stripped of
the quotes that enclose them, have one of each adjacent pair of these quotes removed, and have
(backslash, newline) pairs changed to a single newline character; the results are concatenated to
produce the string passed to the operating system as the description of the process to be started.
Thus for the four pipe functions above, the system sees the commands

```ampl
awk '{print ($1+$2)/2}'
awk '{printf "x%s\n", $1}'
awk '{printf "XX%s\n", $1}'
cat
```

respectively. Function cat illustrates the optional format fmt phrase. If fmt results in a string
that does not end in a newline, AMPL appends a newline character. If no fmt is given, each
numeric argument is converted to the shortest decimal string that rounds to the argument value.

Caution: The line returned by a pipe function must be a complete line, i.e., must end with a
newline character, and the pipe function process must flush its buffers to prevent deadlock. (Pipe
functions do not work with most standard Unix programs, because they don't flush output at the
end of each line.)
Imported functions may be invoked with conventional functional notation, as illustrated above.
In addition, iterated arguments are allowed. More precisely, if f is an imported function, an invocation
of f has the form f(arglist) in which arglist is as for the print and printf commands
— a possibly empty, comma-separated list of expressions and iterated-arglists:

```ampl
ampl: function mean pipe 'awk "{x = 0\
        for(i = 1; i <= NF; i++) x += $i\
        printf "%.17g\n", x/NF}"';
ampl: display mean({i in 1..100} i);
mean({i in 1 .. 100} (i)) = 50.5
ampl: display mean({i in 1..50}(i,i+50));
mean({i in 1 .. 50} (i, i + 50)) = 50.5
ampl: display mean({i in 0..90 by 10}({j in 1..10} i + j));
mean({i in 0 .. 90 by 10} ({j in 1 .. 10} (i + j))) = 50.5
```

The command

```ampl
reset function name opt ;
```

closes all pipe functions, causing them to be restarted if invoked again. If an function is named
explicitly, only that function is closed.